# Notebook 01 — Exploring the Companies KG

Opening moves of the graph-navigation demo: connect to the live [Companies KG](https://demo.neo4jlabs.com)
and let the database describe itself.

Everything below goes through `src/neo4jev/neo4j_access.py`, and that library hardcodes no
label, index, relationship type or property name — all of it is read back from the live
graph (`CALL db.labels()`, `SHOW INDEXES`). This notebook does name a few labels to tour,
but only ones it first discovered in the graph, and it justifies the one index name it
picks. That schema-agnostic habit is what lets the navigator in the later notebooks work on
any graph.

The tour:

1. connect using the settings in `.env`
2. list the graph's labels
3. ask `detect_indexes()` which lookup modes (`exact` / `fulltext` / `vector`) each label supports
4. run one real `search_start_nodes()` lookup per detected mode

In [1]:
from contextlib import ExitStack

from neo4jev.neo4j_access import open_access

_stack = ExitStack()
access = _stack.enter_context(open_access())
print(f"Connected to database {access.database!r}")

Connected to database 'companies'


## 1. Labels

`list_labels()` is a thin wrapper over `CALL db.labels()`. The `_Bloom_*` entries are
Bloom's own scratch labels, and `Fewshot` holds prompt examples rather than business data.
We report them as-is rather than filtering them out: `list_labels()` is a faithful view of
the database, and the labels the demo actually navigates are picked later, on purpose.

In [2]:
labels = access.list_labels()
print(f"{len(labels)} labels")
for label in labels:
    print(" -", label)

10 labels
 - Article
 - Chunk
 - City
 - Country
 - Fewshot
 - IndustryCategory
 - Organization
 - Person
 - _Bloom_Perspective_
 - _Bloom_Scene_


## 2. Which lookup modes does each label support?

* `exact` is always available — it is a plain substring/equality scan over every string
  property of the label, so it needs no index.
* `fulltext` and `vector` are reported **only** when the database actually has a matching
  index for that label, which is why `detect_indexes()` is the thing that decides which
  search modes the UI may offer.

For the vector indexes we also print the declared `vector.dimensions`, since that is what
an embedder has to produce.

In [3]:
labels_of_interest = ["Organization", "Person", "Article", "Chunk"]

indexes_by_label = {label: access.detect_indexes(label) for label in labels_of_interest}

for label, indexes in indexes_by_label.items():
    print(f"{label}: modes = {', '.join(indexes.modes)}")
    for ref in indexes.fulltext + indexes.vector:
        dimensions = f" dims={ref.dimensions}" if ref.dimensions else ""
        print(f"    {ref.kind:<8} {ref.name:<18} props={list(ref.properties)}{dimensions}")

Organization: modes = exact, fulltext
    FULLTEXT entity             props=['name']
Person: modes = exact, fulltext
    FULLTEXT entity             props=['name']
Article: modes = exact
Chunk: modes = exact, fulltext, vector
    FULLTEXT news_fulltext      props=['text']
    VECTOR   news               props=['embedding'] dims=1536
    VECTOR   news_google        props=['embedding_google'] dims=768
    VECTOR   news_google_004    props=['embedding_google_004'] dims=768
    VECTOR   news_sbert         props=['embedding_sbert'] dims=384


Notice that `Article` gets nothing but `exact`, while `Chunk` carries both kinds of index.
The graph therefore already answers "which searches can I offer for this label?" without
any configuration on our side.

In [4]:
def summarize(nodes):
    for node in nodes:
        props = ", ".join(f"{k}={str(v)[:50]!r}" for k, v in list(node.props.items())[:3])
        score = "" if node.score is None else f"  score={node.score:.3f}"
        print(f"  [{'/'.join(node.labels)}] {props}{score}")

## 3. One real lookup per detected mode

### 3a. `exact`

No index involved: the query is a case-insensitive `CONTAINS` test over *every* string
property of each `Organization`, not just its name. That is why `Simply Mac` ("US-based
retail chain focusing on Apple products") legitimately shows up. Whole-property equality is
then used only for ranking, so `Apple` and `Apple Corps` sort above the rest of the match
set.

In [5]:
exact_hits = access.search_start_nodes("Organization", "Apple", "exact", limit=5)
print(f"exact match for 'Apple' -> {len(exact_hits)} hit(s)")
summarize(exact_hits)

exact match for 'Apple' -> 5 hit(s)
  [Organization] summary='American multinational technology company', diffbotId='https://diffbot.com/entity/EHb0_0NEcMwyY8b083taTTw', name='Apple'
  [Organization] summary='', diffbotId='https://diffbot.com/entity/EDZaxeZ69N6KV1ekig3p0-Q', name='Apple'
  [Organization] diffbotId='https://diffbot.com/entity/E3fDQWWYQNL-DO9Ad-BAK3A', name='Apple Corps', id='E3fDQWWYQNL-DO9Ad-BAK3A'
  [Organization] summary='Internet online music service by Apple', diffbotId='https://diffbot.com/entity/E-AdsR-iMOJu3jAkCx0DfeQ', name='Apple Music'
  [Organization] summary='US-based retail chain focusing on Apple products', diffbotId='https://diffbot.com/entity/EwLGN4PGEM_KlVprAe6f8Ew', name='Simply Mac'


### 3b. `fulltext`

`detect_indexes("Organization")` found the `entity` fulltext index (on the `name`
property), so this mode is offered. The query text is Lucene-escaped by
`search_start_nodes()` before it reaches `db.index.fulltext.queryNodes`, so user input is
always treated as literal text.

Two distinct organizations are named exactly `Apple`, and they tie on score — the
relative order of tied hits is not stable, so anything downstream (like notebook 02) must
pick from the returned list rather than trusting `limit=1`.

In [6]:
fulltext_hits = access.search_start_nodes("Organization", "Apple", "fulltext", limit=5)
print(f"fulltext match for 'Apple' -> {len(fulltext_hits)} hit(s)")
summarize(fulltext_hits)

fulltext match for 'Apple' -> 5 hit(s)
  [Organization] summary='', diffbotId='https://diffbot.com/entity/EDZaxeZ69N6KV1ekig3p0-Q', name='Apple'  score=4.502
  [Organization] summary='American multinational technology company', diffbotId='https://diffbot.com/entity/EHb0_0NEcMwyY8b083taTTw', name='Apple'  score=4.502
  [Organization] summary='Organization based in New Milford, Connecticut, Un', diffbotId='https://diffbot.com/entity/E8mgeBGnVO5CsZdGFNcvaNQ', name='Apple Studios'  score=3.618
  [Organization] summary='Organization based in München, Free State of Bavar', diffbotId='https://diffbot.com/entity/EgUo2KRWxOJOMGaK7GJlU3g', name='Apple (Germany)'  score=3.618
  [Organization] summary='Customer support division of Apple Inc.', diffbotId='https://diffbot.com/entity/EGYGV3mkjNQeNyIX3oLPWTA', name='Apple Support'  score=3.618


### 3c. `vector`

`Chunk` has **four** vector indexes with **three** different dimensions (1536 for `news`,
768 for `news_google` / `news_google_004`, 384 for `news_sbert`). Left alone,
`search_start_nodes()` picks the first one in name order; we pass `index_name="news"`
explicitly to make the sample deterministic.

A caveat worth stating plainly: this demo has no embedding-provider credentials, so the
default embedder is a deterministic hash-seeded pseudo-embedding (`default_embedder` in
`neo4j_access.py`). The call exercises the index and returns real chunks in a real order,
but the ranking carries **no semantic meaning**. Pass a real embedder to `open_access()`
to get meaningful neighbours.

In [7]:
vector_hits = access.search_start_nodes(
    "Chunk", "renewable energy", "vector", limit=5, index_name="news"
)
print(f"vector match for 'renewable energy' in index 'news' -> {len(vector_hits)} hit(s)")
summarize(vector_hits)

vector match for 'renewable energy' in index 'news' -> 5 hit(s)
  [Chunk] id='d580e1905306dbe1a1ab80ef998d38a47cbf1491', text='Many fears and uncertainties caused by the coronav', embedding='[0.004374796524643898, -0.03563672676682472, -0.00'  score=0.520
  [Chunk] id='f18c5f7db58383d872b843363d5fe489abccc2d1', text=' cable. Of course, buyng a standard HDMI cable ada', embedding='[-0.0035280620213598013, 0.005586643703281879, -0.'  score=0.520
  [Chunk] embedding_google_004='[0.025694405660033226, -0.07668430358171463, -0.00', embedding_sbert='[-0.032350972294807434, -0.028565995395183563, -0.', id='694d485969578e64c39e1be9f6b01c3cd266d3fe'  score=0.520
  [Chunk] id='1b7cd7ff8663dfe5d398ecf835ca11c0c86cb874', text='Hard Drive Data Destruction\nWhy Is Hard Disk Data ', embedding='[0.013666076585650444, -0.004609428346157074, 0.00'  score=0.520
  [Chunk] embedding_google_004='[-0.016802916303277016, -0.04332969710230827, -0.0', id='abe50751478a7cb1e9b3235de847eb8fdf6008be', text='The rap

### 3d. Fulltext on `Chunk`

`Chunk` also has a fulltext index (`news_fulltext`, over the `text` property), so the same
label supports all three modes at once — the widest search surface in this graph.

In [8]:
chunk_fulltext_hits = access.search_start_nodes("Chunk", "renewable energy", "fulltext", limit=5)
print(f"fulltext match for 'renewable energy' -> {len(chunk_fulltext_hits)} hit(s)")
summarize(chunk_fulltext_hits)

fulltext match for 'renewable energy' -> 5 hit(s)
  [Chunk] id='861067a8fe52fc4b31679faffaa1033eddf9ca9b', text=' its sustainability initiatives. The company is co', embedding='[0.013155626133084297, -0.04098128899931908, -0.03'  score=6.808
  [Chunk] id='95ccd9698e67d3faabf34cf6bbfe0cacea0b0526', text=' development, construction, and operation of renew', embedding='[-0.010368678718805313, -0.027525819838047028, -0.'  score=6.734
  [Chunk] id='a4d7a717875c66c7dd586aadf935e4cac81d8128', text=' experienced professionals in the renewable energy', embedding='[0.019949903711676598, -0.04422919079661369, -0.00'  score=6.725
  [Chunk] id='364a8d701fdcccb3f5443fd8fe6345e68ab0cde2', text='Renewable wind energy refers to the energy that is', embedding='[0.0033644353970885277, -0.03285985067486763, 0.00'  score=6.699
  [Chunk] id='7e8547311b92d8456a6d8cf5a22422c3527c60bc', text='Latest News\nJasmine Energy’s Jasmine RECs renewabl', embedding='[-0.0012946610804647207, -0.012930033728480339, -0'  s

## 4. The detection is real, not cosmetic

Asking for a mode the graph cannot serve fails loudly instead of silently returning
nothing, which is what lets the Streamlit app build its mode picker straight from
`detect_indexes()`.

In [9]:
try:
    access.search_start_nodes("Article", "energy", "fulltext")
except ValueError as exc:
    print(f"Article supports only {indexes_by_label['Article'].modes} -> {exc}")

Article supports only ('exact',) -> No fulltext index available for label 'Article'


## Next

Notebook 02 (`02_navigator_dry_run.ipynb`) picks a start node with the lookup modes shown
here, fetches its outgoing relationships, and runs a single TypeSafe `Choice` + `Noul` hop
against it.

In [10]:
_stack.close()
print("Connection closed.")

Connection closed.
